# 08 — Error Analysis, Customer Personas & Conclusion

Understand **who** the model gets wrong and **why**, build customer personas, define a recommendation engine, and state limitations.

## Setup & Load

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import os, pickle, json

os.makedirs('../outputs', exist_ok=True)

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
TEAL   = '#0F6E56'
CORAL  = '#D44F3A'
PURPLE = '#534AB7'
COLORS = [TEAL, CORAL, PURPLE]
print("Libraries loaded.")


Libraries loaded.


In [2]:
X_train        = pd.read_csv('../data/X_train.csv')
X_test         = pd.read_csv('../data/X_test.csv')
X_train_scaled = pd.read_csv('../data/X_train_scaled.csv')
X_test_scaled  = pd.read_csv('../data/X_test_scaled.csv')
y_train        = pd.read_csv('../data/y_train.csv').squeeze()
y_test         = pd.read_csv('../data/y_test.csv').squeeze()
print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print(f"Train churn rate: {y_train.mean():.3f} | Test churn rate: {y_test.mean():.3f}")


Train: (5634, 26) | Test: (1409, 26)
Train churn rate: 0.265 | Test churn rate: 0.265


In [3]:
logit_sk  = pickle.load(open('../data/model1_logistic.pkl', 'rb'))
rf_model  = pickle.load(open('../data/model2_rf.pkl',       'rb'))
xgb_tuned = pickle.load(open('../data/model3_xgb.pkl',      'rb'))
best_params = json.load(open('../data/xgb_best_params.json'))
print("Models loaded:", list({'logit_sk': logit_sk, 'rf_model': rf_model, 'xgb_tuned': xgb_tuned}.keys()))


Models loaded: ['logit_sk', 'rf_model', 'xgb_tuned']


In [4]:
CHOSEN_THRESHOLD = 0.35
probs_xgb   = xgb_tuned.predict_proba(X_test)[:, 1]
preds_final = (probs_xgb >= CHOSEN_THRESHOLD).astype(int)

df_orig = pd.read_csv('../data/telco_churn.csv')
df_orig['TotalCharges'] = pd.to_numeric(df_orig['TotalCharges'], errors='coerce').fillna(0)
df_orig['Churn_binary'] = df_orig['Churn'].map({'Yes': 1, 'No': 0})

for col in ['OnlineSecurity','OnlineBackup','DeviceProtection',
            'TechSupport','StreamingTV','StreamingMovies']:
    df_orig[col] = df_orig[col].map({'Yes': 1, 'No': 0,
                                     'No phone service': 0, 'No internet service': 0})
service_cols = ['OnlineSecurity','OnlineBackup','DeviceProtection',
                'TechSupport','StreamingTV','StreamingMovies']
df_orig['num_services'] = df_orig[service_cols].sum(axis=1)
df_orig['tenure_group'] = pd.cut(df_orig['tenure'],
    bins=[0, 12, 36, 72], labels=['New (0-12m)', 'Mid (13-36m)', 'Loyal (37-72m)'])

print("Setup complete.")


Setup complete.


## False Negative & False Positive Analysis

In [5]:
fn_mask = (preds_final == 0) & (y_test.values == 1)  # missed churners
fp_mask = (preds_final == 1) & (y_test.values == 0)  # wrongly flagged

fn_df = X_test[fn_mask].copy()
fp_df = X_test[fp_mask].copy()

print(f"False Negatives (missed churners)  : {fn_mask.sum()}")
print(f"False Positives (wrongly flagged)  : {fp_mask.sum()}")
print()
cols_profile = ['tenure', 'MonthlyCharges', 'num_services', 'charges_per_month']
print("FN PROFILE (customers the model failed to flag as churners):")
print(fn_df[cols_profile].describe().round(2).loc[['mean', '50%']])
print()
print("FP PROFILE (loyal customers wrongly flagged as churners):")
print(fp_df[cols_profile].describe().round(2).loc[['mean', '50%']])


False Negatives (missed churners)  : 104
False Positives (wrongly flagged)  : 214

FN PROFILE (customers the model failed to flag as churners):
      tenure  MonthlyCharges  num_services  charges_per_month
mean   30.28           63.90          2.22              60.17
50%    28.00           60.12          2.00              55.46

FP PROFILE (loyal customers wrongly flagged as churners):
      tenure  MonthlyCharges  num_services  charges_per_month
mean   17.81           77.75          1.71              68.21
50%    12.00           80.93          2.00              73.54


**Interpretation**: Compare FN profile to FP profile. High-tenure FNs suggest the model misses churners with long tenure (contract expiry surprise). Low-MonthlyCharges FPs suggest some new customers are flagged despite low charges.

## Customer Personas

In [6]:
print("=" * 60)
print("  CUSTOMER PERSONAS — from churner profiles by tenure group")
print("=" * 60)

for group, label in [('New (0-12m)',    'The Frustrated New Customer'),
                     ('Mid (13-36m)',   'The Price-Sensitive Mid-Tenure Customer'),
                     ('Loyal (37-72m)', 'The Silent Long-Tenure Churner')]:
    mask_g   = df_orig['tenure_group'] == group
    churners = df_orig[mask_g & (df_orig['Churn_binary'] == 1)]
    if len(churners) == 0:
        continue
    print(f"\nPersona: '{label}'")
    print(f"  Count               : {len(churners)}")
    print(f"  Avg tenure          : {churners['tenure'].median():.0f} months")
    print(f"  Avg monthly charges : ${churners['MonthlyCharges'].median():.0f}")
    print(f"  Avg num_services    : {churners['num_services'].median():.0f}")
    print(f"  Typical contract    : {churners['Contract'].mode()[0]}")
    print(f"  Internet service    : {churners['InternetService'].mode()[0]}")
    print(f"  Payment method      : {churners['PaymentMethod'].mode()[0]}")
    print(f"  Has Tech Support    : {df_orig.loc[churners.index,'TechSupport'].mode()[0] if 'TechSupport' in df_orig.columns else 'N/A'}")


  CUSTOMER PERSONAS — from churner profiles by tenure group

Persona: 'The Frustrated New Customer'
  Count               : 1037
  Avg tenure          : 3 months
  Avg monthly charges : $74
  Avg num_services    : 1
  Typical contract    : Month-to-month
  Internet service    : Fiber optic
  Payment method      : Electronic check
  Has Tech Support    : 0

Persona: 'The Price-Sensitive Mid-Tenure Customer'
  Count               : 474
  Avg tenure          : 22 months
  Avg monthly charges : $85
  Avg num_services    : 2
  Typical contract    : Month-to-month
  Internet service    : Fiber optic
  Payment method      : Electronic check
  Has Tech Support    : 0

Persona: 'The Silent Long-Tenure Churner'
  Count               : 358
  Avg tenure          : 52 months
  Avg monthly charges : $97
  Avg num_services    : 3
  Typical contract    : Month-to-month
  Internet service    : Fiber optic
  Payment method      : Electronic check
  Has Tech Support    : 0


**Persona Interpretations & Targeting Strategy:**

| Persona | Key Signal | Recommended Action |
|---|---|---|
| **The Frustrated New Customer** (0–12m) | 3 months tenure, $74/month, no Tech Support, Fiber | Immediate outreach within first 60 days. Offer free Tech Support trial — service quality is the trigger, not price. |
| **The Price-Sensitive Mid-Tenure Customer** (13–36m) | 22 months tenure, $85/month, month-to-month | Contract upgrade offer (15% discount to switch to 1-year). They have enough loyalty to be retained if given a reason to stay. |
| **The Silent Long-Tenure Churner** (37–72m) | 52 months tenure, $97/month, month-to-month | Highest-value, hardest to detect. Likely triggered by contract expiry or competitor offer. Assign account manager + loyalty reward — price discounts alone won't work here. |

**Key insight**: All three personas share the same payment method (Electronic check) and lack of Tech Support. These are two actionable levers the business can address immediately — independent of the model score.

## Recommendation Engine

The recommendation engine translates a raw churn probability into a **specific, actionable retention instruction** for a customer service agent. 

**Design philosophy:**
- Three risk tiers: LOW (<40%), MEDIUM (40–70%), HIGH (>70%) — maps to urgency of response
- Actions are ranked by cost: free service offers first, discounts second, personal outreach third
- Rules are derived directly from the SHAP interaction analysis in Notebook 05 (tenure × charges × tech support)
- The function is deliberately rule-based rather than a second model — interpretability matters for agent trust

**Four action types and their business rationale:**
1. **No intervention** — below threshold, intervention cost exceeds expected value
2. **Free Tech Support offer** — targets new high-charge customers where service quality (not price) is the driver
3. **Contract upgrade discount** — targets high-charge month-to-month customers where lock-in is the lever
4. **Account manager + loyalty reward** — targets long-tenure surprise churners where relationship matters more than price

In [7]:
def recommend_retention_action(idx, X_df, probs_arr, threshold=0.35):
    prob    = probs_arr[idx]
    row     = X_df.iloc[idx]
    level   = 'HIGH' if prob > 0.70 else 'MEDIUM' if prob > 0.40 else 'LOW'
    tenure  = row.get('tenure', 0)
    monthly = row.get('MonthlyCharges', 0)
    m2m     = (row.get('Contract_One year', 0) == 0 and row.get('Contract_Two year', 0) == 0)
    tech    = row.get('TechSupport', 0)
    services = row.get('num_services', 0)

    print(f"Customer #{idx} | Churn Probability: {prob:.1%} | Risk: {level}")
    print(f"  Tenure: {tenure:.0f}m | MonthlyCharges: ${monthly:.0f} | Services: {services:.0f}")
    if prob < 0.40:
        print("  ACTION: No intervention needed. Monitor quarterly.")
    elif tenure <= 12 and not tech and monthly > 65:
        print("  ACTION: Offer FREE Tech Support for 3 months.")
        print("  REASON: New customer, high charges, no support — service quality issue.")
    elif monthly > 70 and m2m:
        print("  ACTION: Offer 15% discount to upgrade to 1-year contract.")
        print("  REASON: High-paying month-to-month — contract upgrade reduces churn.")
    elif tenure > 36 and prob > 0.60:
        print("  ACTION: Assign dedicated account manager + loyalty reward.")
        print("  REASON: Long-tenure surprise churner — needs personal attention.")
    else:
        print("  ACTION: Proactive outreach call + personalised service review.")
    print()

# Test on one customer from each risk level
tp_indices = np.where((preds_final == 1) & (y_test.values == 1))[0]
low_idx    = np.where(probs_xgb < 0.4)[0][0]
mid_idx    = np.where((probs_xgb >= 0.4) & (probs_xgb < 0.7))[0][0]
high_idx   = tp_indices[0]

print("RECOMMENDATION ENGINE — Sample Outputs")
print("=" * 60)
for idx in [low_idx, mid_idx, high_idx]:
    recommend_retention_action(idx, X_test, probs_xgb)


RECOMMENDATION ENGINE — Sample Outputs
Customer #0 | Churn Probability: 3.1% | Risk: LOW
  Tenure: 72m | MonthlyCharges: $114 | Services: 6
  ACTION: No intervention needed. Monitor quarterly.

Customer #5 | Churn Probability: 53.2% | Risk: MEDIUM
  Tenure: 21m | MonthlyCharges: $100 | Services: 3
  ACTION: Offer 15% discount to upgrade to 1-year contract.
  REASON: High-paying month-to-month — contract upgrade reduces churn.

Customer #9 | Churn Probability: 35.1% | Risk: LOW
  Tenure: 17m | MonthlyCharges: $45 | Services: 0
  ACTION: No intervention needed. Monitor quarterly.



## Two Systematic Failure Modes

### Failure Mode 1 — Long-Tenure Contract Expiry (False Negatives)

The model **consistently misses** loyal customers (tenure > 36m) on two-year contracts whose contract has just expired. The model has learned from history that high-tenure = safe — it cannot detect the expiring-contract signal without a renewal date feature.

**Root Cause**: Contract renewal date is not in the dataset.

**Mitigation Proxy** (no renewal date available):
```python
# Proxy: customers whose estimated contract age is 22-24 months
# (approaching renewal on a 24-month cycle)
df['contract_age_proxy'] = df['tenure'] % 24
df['near_contract_expiry'] = (df['contract_age_proxy'] >= 22).astype(int)
```
Customers with `near_contract_expiry = 1` should receive **proactive outreach independent of model score** — they are structurally vulnerable.

---

### Failure Mode 2 — New High-Value Customers (False Positives)

The model flags new customers (tenure < 6m) with high monthly charges as churn risks, but many stay. These customers are in a **genuine trial phase** — their intent cannot be read from behaviour data.

**Root Cause**: Short tenure makes the model extrapolate from the high-risk population statistics, when many of these customers are simply new.

**Mitigation**:
```python
# Apply a higher threshold for trial-phase customers (tenure < 3m)
TRIAL_THRESHOLD = 0.55  # stricter than standard 0.35
df['trial_phase'] = (df['tenure'] < 3).astype(int)
```
Stratified threshold tuning by tenure group addresses this **without retraining**.


## Section 10 — Limitations & Critical Thinking

### Limitation 1 — Missing Behavioural Data (Most Critical)
No record of customer service calls, complaints, network outage experiences, or competitor interactions. Complaint history is the single strongest predictor of churn in real deployments — its absence is the primary reason ~40% of churners go undetected.
→ *Future work*: Integrate CRM complaint logs and call centre data.

### Limitation 2 — Retention Success Rate Assumed Constant (30%)
Our ROI calculation assumes 30% of flagged churners can be retained. In reality this varies by offer type, segment, and timing. If actual rate is 15%, all profit estimates halve.
→ *Bound estimates*: Run ROI at 15%, 20%, 25%, 30% (done in Notebook 06 sensitivity analysis).

### Limitation 3 — No Contract Renewal Date
The single most predictive event for loyal-customer churn is contract expiry — which is absent from this dataset. This is the direct cause of Failure Mode 1.
→ *Mitigation*: Use `contract_age_proxy = tenure % 24` as a proxy.

### Limitation 4 — Static Cross-Sectional Dataset
All 7,043 rows are independent customer observations at one point in time. The model cannot learn from **trajectories**:
- A customer whose charges *increased* $20 last month is far riskier than one with the same current charges but stable history.
- A customer who *downgraded* from 5 to 2 services shows disengagement.
→ *Future work*: Engineer delta features: `charge_change_last3m`, `services_dropped_last6m`.


In [8]:
from sklearn.metrics import roc_auc_score, recall_score, fbeta_score, confusion_matrix

df_raw = pd.read_csv('../data/telco_churn.csv')
df_raw['TotalCharges'] = pd.to_numeric(df_raw['TotalCharges'], errors='coerce').fillna(0)
monthly_rev    = df_raw['MonthlyCharges'].mean()
annual_rev     = monthly_rev * 12
retention_cost = 10

gains_df = pd.DataFrame({'prob': probs_xgb, 'actual': y_test.values})              .sort_values('prob', ascending=False).reset_index(drop=True)
total_churners  = gains_df['actual'].sum()
gains_df['pct_churners_captured'] = gains_df['actual'].cumsum() / total_churners * 100

profit_rows = []
for pct in np.arange(1, 101, 1):
    n  = int(pct / 100 * len(gains_df))
    t  = gains_df.iloc[:n]
    tp = t['actual'].sum()
    fp = n - tp
    profit_rows.append({'pct': pct, 'profit': tp*(0.30*annual_rev-retention_cost) - fp*retention_cost})
profit_df    = pd.DataFrame(profit_rows)
best_row     = profit_df.loc[profit_df['profit'].idxmax()]
best_pct     = best_row['pct']
best_profit  = best_row['profit']
top10_pct    = gains_df['pct_churners_captured'].iloc[min(int(0.10*len(gains_df)), len(gains_df)-1)]

print("=" * 65)
print("  ★ FINAL SUMMARY")
print("=" * 65)
print(f"Final Model      : XGBoost (tuned with GridSearchCV)")
print(f"Test AUC         : {roc_auc_score(y_test, probs_xgb):.3f}")
print(f"Test F2 Score    : {fbeta_score(y_test, preds_final, beta=2):.3f}  (β=2, recall-weighted)")
print(f"Test Recall      : {recall_score(y_test, preds_final):.3f}")
print(f"Chosen Threshold : {CHOSEN_THRESHOLD}")
print(f"Top 10% Gains    : {top10_pct:.1f}% of churners captured ({top10_pct/10:.1f}x lift)")
print(f"Profit-optimal   : Contact top {best_pct:.0f}% → ${best_profit:,.0f} campaign profit (test set)")
print()
print("THREE KEY FINDINGS:")
print("  1. Contract type is the master lever: month-to-month 43% churn vs two-year 3%.")
print("     → Immediate priority: convert month-to-month customers to one-year contracts.")
print("  2. Price sensitivity is a new-customer problem only.")
print("     → Discounts should target high-charges customers in first 12 months — not loyal ones.")
print("  3. The model tells you who to call and in what order.")
print(f"     → Top 10% by score = {top10_pct:.1f}% of churners. Contact top {best_pct:.0f}% for max profit.")
print()
print("21 output charts saved to ../outputs/")
print("3 models saved   : model1_logistic.pkl | model2_rf.pkl | model3_xgb.pkl")


  ★ FINAL SUMMARY
Final Model      : XGBoost (tuned with GridSearchCV)
Test AUC         : 0.845
Test F2 Score    : 0.682  (β=2, recall-weighted)
Test Recall      : 0.722
Chosen Threshold : 0.35
Top 10% Gains    : 27.8% of churners captured (2.8x lift)
Profit-optimal   : Contact top 77% → $75,889 campaign profit (test set)

THREE KEY FINDINGS:
  1. Contract type is the master lever: month-to-month 43% churn vs two-year 3%.
     → Immediate priority: convert month-to-month customers to one-year contracts.
  2. Price sensitivity is a new-customer problem only.
     → Discounts should target high-charges customers in first 12 months — not loyal ones.
  3. The model tells you who to call and in what order.
     → Top 10% by score = 27.8% of churners. Contact top 77% for max profit.

21 output charts saved to ../outputs/
3 models saved   : model1_logistic.pkl | model2_rf.pkl | model3_xgb.pkl


## Project Conclusion

This project framed churn prediction as a **decision problem under asymmetric costs**, not a classification accuracy problem. The key contributions are:

1. **Cost-aware threshold selection** — the 0.35 threshold was chosen by maximising Expected Value (FN cost = $780, FP cost = $10, ratio 78:1), not by maximising F1. A naive threshold of 0.50 would have left ~40 additional churners undetected per campaign cycle.

2. **SHAP-derived business insight** — the tenure × MonthlyCharges interaction (Notebook 05) revealed that price sensitivity is exclusively a new-customer phenomenon. This prevents the business from wasting discount budget on loyal customers who were never going to leave.

3. **Actionable segmentation** — the stratified analysis (Notebook 07) confirmed that a single model fits all segments adequately, but the recommendation engine applies segment-specific interventions based on the underlying driver (service quality vs contract lock-in vs loyalty).

4. **Honest limitations** — the absence of complaint history, contract renewal dates, and temporal trajectories are the primary ceiling on this model's performance. These are data collection problems, not modelling problems. The next highest-value investment is enriching the feature set, not tuning the model further.